# VERSION AMÉLIORÉE — Comparaison Isolation Forest vs DBSCAN

**Documents sources :** `02_isolation_forest.ipynb` et `03_dbscan_afdm.ipynb`  
**Objectif :** comparer les résultats réellement obtenus et recommander le modèle approprié selon le besoin.

> **Conclusion :** DBSCAN est actuellement préférable pour produire une liste courte d’anomalies à investiguer. Isolation Forest reste plus approprié pour un futur scoring en temps réel, mais seulement après recalibration. Aucun des deux modèles ne doit actuellement bloquer automatiquement une transaction.

## 1. Résultats relevés dans les notebooks 02 et 03

| Indicateur | Isolation Forest | DBSCAN |
|---|---:|---:|
| Données évaluées | Test temporel : 110 000 | Jeu complet : 550 000 |
| Anomalies signalées | 108 550 | 3 922 |
| Taux d’alertes | 98,68 % | 0,71 % |
| Précision | 0,90 % | 5,63 % |
| Rappel | 99,09 % | 4,54 % |
| F1-score | 1,78 % | 5,03 % |
| AUC-ROC | 0,5411 | Non calculée |
| PR-AUC | 0,0110 | Non calculée |
| Résultat complémentaire | 107 573 faux positifs | 579 clusters ; silhouette = 0,9203 |

### Interprétation

- **Isolation Forest** retrouve presque toutes les fraudes, mais uniquement parce que le seuil retenu classe 98,68 % des transactions comme anomalies. Ce résultat est inexploitable opérationnellement.
- **DBSCAN** produit beaucoup moins d’alertes et obtient une précision environ 6,3 fois supérieure. Cependant, son rappel de 4,54 % signifie qu’il manque la grande majorité des fraudes.
- Le score de silhouette élevé de DBSCAN mesure la séparation géométrique des clusters. Il ne démontre pas que les anomalies correspondent aux fraudes.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

resultats = pd.DataFrame({
    'Modèle': ['Isolation Forest', 'DBSCAN'],
    'Précision (%)': [0.90, 5.6349],
    'Rappel (%)': [99.09, 4.5352],
    'F1-score (%)': [1.78, 5.0256],
    "Taux d'alertes (%)": [98.68, 0.7131]
}).set_index('Modèle')

display(resultats.round(2))
ax = resultats.plot(kind='bar', figsize=(11, 6), width=0.75)
ax.set_title('Comparaison des résultats observés')
ax.set_ylabel('Pourcentage')
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

## 2. Comparaison fonctionnelle

| Critère | Isolation Forest | DBSCAN |
|---|---|---|
| Principe | Isolation aléatoire par arbres | Regroupement selon la densité locale |
| Sortie | Score continu ajustable par seuil | Cluster ou bruit (`-1`) |
| Nouvelle transaction | Scoring natif avec `decision_function` | Pas de prédiction native dans `sklearn.DBSCAN` |
| Temps réel | Adapté | Peu adapté |
| Gros volumes | Bonne scalabilité | Coût mémoire et calcul plus élevé |
| Paramètres sensibles | Seuil, contamination, échantillonnage | `eps`, `min_samples`, métrique, représentation FAMD |
| Usage principal | Priorisation des alertes | Segmentation et découverte de comportements atypiques |
| Limite principale | Faux positifs si seuil mal calibré | Difficulté avec les densités variables et les nouveaux points |


## 3. Limites de la comparaison

Les résultats ne constituent pas encore une comparaison expérimentale parfaitement équitable :

1. Isolation Forest est évalué sur un test temporel de 110 000 transactions, alors que DBSCAN est entraîné et évalué sur le jeu complet de 550 000 transactions.
2. Les métriques DBSCAN sont donc potentiellement optimistes, car les mêmes observations servent à former et à évaluer les clusters.
3. Le seuil d’Isolation Forest est incohérent avec une utilisation réelle : il déclenche une alerte sur presque toutes les transactions.
4. Les métriques internes de clustering ne sont pas des métriques de détection de fraude.
5. Une anomalie statistique n’est pas automatiquement une fraude, et une fraude peut ressembler à une opération normale.

Le F1 plus élevé de DBSCAN indique seulement qu’il offre le meilleur compromis **dans les exécutions actuelles**.